# Toggle Switch

In this example notebook, we build a bistable toggle switch model with BioCRNpyler, export it to SBML, and load the SBML model into AutoReduce to obtain a reduced model that is suitable for design and analysis.


In [1]:
from pathlib import Path
import numpy as np
from autoreduce import load_sbml, solve_ode

In [2]:
from biocrnpyler import Species
from biocrnpyler.components import DNAassembly, RegulatedPromoter, DNABindingSite
from biocrnpyler.mechanisms.global_mechanisms import Dilution
from biocrnpyler.mixtures import SimpleTxTlExtract

parameter_file = Path("models/design1_parameters.tsv")
sbml_file = Path("models/design1_regulated_promoter.xml")


## Build bistable toggle switch model using BioCRNpyler

The design has two regulated promoters. Each promoter has one activating regulator and one repressing regulator, and the resulting CRN is exported as SBML for AutoReduce.


In [3]:
A1 = Species("A1", material_type="protein")
A2 = Species("A2", material_type="protein")
R1 = Species("R1", material_type="protein")
R2 = Species("R2", material_type="protein")

# Activator promoter only:
# gives D1 leak transcription and A1-bound transcription.
p1 = RegulatedPromoter(
    name="p1",
    regulators=[A1],
    leak=True,
)

# gives D2 leak transcription and A2-bound transcription.
p2 = RegulatedPromoter(
    name="p2",
    regulators=[A2],
    leak=True,
)

D1 = DNAassembly(
    name="D1",
    promoter=p1,
    rbs="utr1",
    transcript="m1",
    protein=[A1, R1],
)

D2 = DNAassembly(
    name="D2",
    promoter=p2,
    rbs="utr1",
    transcript="m2",
    protein=[A2, R2],
)

# Repressor binding only, no transcription.
# This gives R2 + D1 <--> D1:R2 and R1 + D2 <--> D2:R1.
R2_binds_D1 = DNABindingSite(
    name="R2_binds_D1",
    binders=R2,
)
R2_binds_D1.dna_to_bind = D1.dna

R1_binds_D2 = DNABindingSite(
    name="R1_binds_D2",
    binders=R1,
)
R1_binds_D2.dna_to_bind = D2.dna

protein_degradation = Dilution(
    name="protein_degradation",
    filter_dict={"protein": True, "complex": False},
    default_on=False,
)

mixture = SimpleTxTlExtract(
    name="Design1_RegulatedPromoter",
    components=[D1, D2, R2_binds_D1, R1_binds_D2],
    parameter_file=str(parameter_file),
    overwrite_parameters=True,
    global_mechanisms={
        "protein_degradation": protein_degradation,
    },
)

initial_conditions = {
    D1.dna: 1.0,
    D2.dna: 1.0,
}

crn = mixture.compile_crn(initial_concentration_dict=initial_conditions)

print(crn.pretty_print(show_rates=True, show_keys=True))

crn.write_sbml_file(sbml_file)
print(f"SBML written to {sbml_file}")

Species(N = 12) = {
    dna[D2] (@ 1.0),  
    dna[D1] (@ 1.0),  
    rna[m2] (@ 0),  
    rna[m1] (@ 0),  
    complex[dna[D2]:protein[R1]] (@ 0),  
    complex[dna[D2]:protein[A2]] (@ 0),  
    complex[dna[D1]:protein[R2]] (@ 0),  
    complex[dna[D1]:protein[A1]] (@ 0),  
    protein[R2] (@ 0),  
    protein[R1] (@ 0),  
    protein[A2] (@ 0),  
    protein[A1] (@ 0),  
}

Reactions (16) = [
0. dna[D1] --> dna[D1]+rna[m1]
 Kf=k_forward * dna_D1
  k_forward=0.25
  found_key=(mech=transcription, partid=p1_leak, name=ktx).
  search_key=(mech=simple_transcription, partid=['p1_leak', None], name=ktx).

1. protein[A1]+dna[D1] <--> complex[dna[D1]:protein[A1]]
 Kf=k_forward * protein_A1 * dna_D1
 Kr=k_reverse * complex_dna_D1_protein_A1_
  k_forward=1.0
  found_key=(mech=one_step_cooperative_binding, partid=p1_A1, name=kb).
  search_key=(mech=one_step_cooperative_binding, partid=['p1_A1', 'dna_protein', None], name=kb).
  k_reverse=1.0
  found_key=(mech=one_step_cooperative_binding, partid

c:\Users\ayush\anaconda3\envs\autoreduce\Lib\site-packages\biocrnpyler\mechanisms\global_mechanisms.py:204: UserWarning: species complex_dna_D1_protein_R2_ has multiple attributes (or material type) which conflict with global mechanism filter {repr(self)}. Using default value False.
  warn(
c:\Users\ayush\anaconda3\envs\autoreduce\Lib\site-packages\biocrnpyler\mechanisms\global_mechanisms.py:204: UserWarning: species complex_dna_D2_protein_A2_ has multiple attributes (or material type) which conflict with global mechanism filter {repr(self)}. Using default value False.
  warn(
c:\Users\ayush\anaconda3\envs\autoreduce\Lib\site-packages\biocrnpyler\mechanisms\global_mechanisms.py:204: UserWarning: species complex_dna_D1_protein_A1_ has multiple attributes (or material type) which conflict with global mechanism filter {repr(self)}. Using default value False.
  warn(
c:\Users\ayush\anaconda3\envs\autoreduce\Lib\site-packages\biocrnpyler\mechanisms\global_mechanisms.py:204: UserWarning: spe

## Load the SBML model into AutoReduce

Using `load_sbml`, we can load the SBML model into AutoReduce. The function returns a `System` object that contains the state variables, parameters, and equations of the system.

In [4]:
sbml_file = "models/design1_regulated_promoter.xml"

sys0 = load_sbml(
    sbml_file,
    outputs=["protein_A1", "protein_A2"],
)

## Inspect the imported SBML

The state names come from the SBML species IDs generated by BioCRNpyler. Inspecting them makes it easier to choose outputs or reduction assumptions in later cells.


In [5]:
print("Full model states:")
for x in sys0.x:
    print("  ", x)

Full model states:
   dna_D1
   rna_m1
   protein_A1
   protein_R1
   complex_dna_D1_protein_A1_
   dna_D2
   rna_m2
   protein_A2
   protein_R2
   complex_dna_D2_protein_A2_
   complex_dna_D1_protein_R2_
   complex_dna_D2_protein_R1_


In [6]:
print("\nFull model parameters:")
for p, v in zip(sys0.params, sys0.params_values):
    print(f"  {p}: {v}")



Full model parameters:
  ktx_p1_leak_transcription: 0.25
  kb_p1_A1_one_step_cooperative_binding: 1.0
  ku_p1_A1_one_step_cooperative_binding: 1.0
  ktx_p1_A1_transcription: 2.0
  ktl_utr1_simple_translation: 5.0
  ktx_p2_leak_transcription: 0.25
  kb_p2_A2_one_step_cooperative_binding: 1.0
  ku_p2_A2_one_step_cooperative_binding: 1.0
  ktx_p2_A2_transcription: 2.0
  kb_R2_one_step_cooperative_binding: 1.0
  ku_R2_one_step_cooperative_binding: 1.0
  kb_R1_one_step_cooperative_binding: 1.0
  ku_R1_one_step_cooperative_binding: 1.0
  kdil_protein_R2_protein_degradation: 1.0
  kdil_protein_A2_protein_degradation: 1.0
  kdil_protein_R1_protein_degradation: 1.0
  kdil_protein_A1_protein_degradation: 1.0
  kdil_rna_m1_rna_degradation: 1.0
  kdil_rna_m2_rna_degradation: 1.0


### Rename species and parameters
For simpler names and to avoid long Sympy names, we rename the (long) BioCRNpyler-generated species and parameters to shorter names. We start by creating sympy symbols.

In [7]:
from sympy import symbols, Symbol, simplify, pprint

D1, D2 = symbols("D1 D2")
m1, m2 = symbols("m1 m2")
A1, A2 = symbols("A1 A2")
R1, R2 = symbols("R1 R2")
C_A1_D1, C_R2_D1, C_A2_D2, C_R1_D2 = symbols(
    "C_A1_D1 C_R2_D1 C_A2_D2 C_R1_D2"
)

ktx0_1, kb_1, ku_1, ktx_1, ktl_1 = symbols(
    "ktx0_1 kb_1 ku_1 ktx_1 ktl_1"
)
ktx0_2, kb_2, ku_2, ktx_2 = symbols(
    "ktx0_2 kb_2 ku_2 ktx_2"
)

kb_R1, ku_R1, kb_R2, ku_R2 = symbols(
    "kb_R1 ku_R1 kb_R2 ku_R2"
)

kdil_R1, kdil_A1, kdil_R2, kdil_A2 = symbols(
    "kdil_R1 kdil_A1 kdil_R2 kdil_A2"
)

kdil_m1, kdil_m2 = symbols("kdil_m1 kdil_m2")


state_rename = {
    Symbol("dna_D1"): D1,
    Symbol("dna_D2"): D2,
    Symbol("rna_m1"): m1,
    Symbol("rna_m2"): m2,
    Symbol("protein_A1"): A1,
    Symbol("protein_A2"): A2,
    Symbol("protein_R1"): R1,
    Symbol("protein_R2"): R2,
    Symbol("complex_dna_D1_protein_A1_"): C_A1_D1,
    Symbol("complex_dna_D1_protein_R2_"): C_R2_D1,
    Symbol("complex_dna_D2_protein_A2_"): C_A2_D2,
    Symbol("complex_dna_D2_protein_R1_"): C_R1_D2,
}

params_rename = {
    Symbol("ktx_p1_leak_transcription"): ktx0_1,
    Symbol("kb_p1_A1_one_step_cooperative_binding"): kb_1,
    Symbol("ku_p1_A1_one_step_cooperative_binding"): ku_1,
    Symbol("ktx_p1_A1_transcription"): ktx_1,
    Symbol("ktl_utr1_simple_translation"): ktl_1,
    Symbol("ktx_p2_leak_transcription"): ktx0_2,
    Symbol("kb_p2_A2_one_step_cooperative_binding"): kb_2,
    Symbol("ku_p2_A2_one_step_cooperative_binding"): ku_2,
    Symbol("ktx_p2_A2_transcription"): ktx_2,
    Symbol("kb_R2_one_step_cooperative_binding"): kb_R2,
    Symbol("ku_R2_one_step_cooperative_binding"): ku_R2,
    Symbol("kb_R1_one_step_cooperative_binding"): kb_R1,
    Symbol("ku_R1_one_step_cooperative_binding"): ku_R1,
    Symbol("kdil_protein_R1_protein_degradation"): kdil_R1,
    Symbol("kdil_protein_A1_protein_degradation"): kdil_A1,
    Symbol("kdil_protein_R2_protein_degradation"): kdil_R2,
    Symbol("kdil_protein_A2_protein_degradation"): kdil_A2,
    Symbol("kdil_rna_m2_rna_degradation"): kdil_m2,
    Symbol("kdil_rna_m1_rna_degradation"): kdil_m1
}

sys0.x = [state_rename.get(x, x) for x in sys0.x]
sys0.params = [params_rename.get(p, p) for p in sys0.params]
sys0.f = [simplify(fi.xreplace(state_rename)) for fi in sys0.f]
sys0.f = [simplify(fi.xreplace(params_rename)) for fi in sys0.f]

print("\nRenamed full system:")
for x, fi in zip(sys0.x, sys0.f):
    print(f"d{x}/dt =")
    pprint(simplify(fi))
    print()


Renamed full system:
dD1/dt =
-A₁⋅D₁⋅kb₁ + C_A1_D1⋅ku₁ + C_R2_D1⋅ku_R2 - D₁⋅R₂⋅kb_R2

dm1/dt =
C_A1_D1⋅ktx₁ + D₁⋅ktx₀ ₁ - kdilₘ₁⋅m₁

dA1/dt =
-A₁⋅D₁⋅kb₁ - A₁⋅kdil_A1 + C_A1_D1⋅ku₁ + ktl₁⋅m₁

dR1/dt =
C_R1_D2⋅ku_R1 - D₂⋅R₁⋅kb_R1 - R₁⋅kdil_R1 + ktl₁⋅m₁

dC_A1_D1/dt =
A₁⋅D₁⋅kb₁ - C_A1_D1⋅ku₁

dD2/dt =
-A₂⋅D₂⋅kb₂ + C_A2_D2⋅ku₂ + C_R1_D2⋅ku_R1 - D₂⋅R₁⋅kb_R1

dm2/dt =
C_A2_D2⋅ktx₂ + D₂⋅ktx₀ ₂ - kdilₘ₂⋅m₂

dA2/dt =
-A₂⋅D₂⋅kb₂ - A₂⋅kdil_A2 + C_A2_D2⋅ku₂ + ktl₁⋅m₂

dR2/dt =
C_R2_D1⋅ku_R2 - D₁⋅R₂⋅kb_R2 - R₂⋅kdil_R2 + ktl₁⋅m₂

dC_A2_D2/dt =
A₂⋅D₂⋅kb₂ - C_A2_D2⋅ku₂

dC_R2_D1/dt =
-C_R2_D1⋅ku_R2 + D₁⋅R₂⋅kb_R2

dC_R1_D2/dt =
-C_R1_D2⋅ku_R1 + D₂⋅R₁⋅kb_R1



## Apply conservation laws

The `solve_conservation_laws` function can be used to eliminate conserved species from the system. This AutoReduce method can find the conserved sets automatically by performing a combinatorial search over the stoichiometry matrix. The user can also specify the conserved sets manually, as we do here for the toggle switch model. The conserved species are the DNA species, which are not degraded in the system. The total amount of each DNA species is conserved.  


In [8]:
from autoreduce import solve_conservation_laws

D1_tot, D2_tot = symbols("D1_tot D2_tot")


sys_cons = solve_conservation_laws(
    sys0,
    total_quantities={
        "D1_tot": 1.0,
        "D2_tot": 1.0,
    },
    conserved_sets=[
        [D1, C_A1_D1, C_R2_D1],
        [D2, C_A2_D2, C_R1_D2],
    ],
    states_to_eliminate=[D1, D2],
    debug=True,
)

sys_cons.f = [simplify(fi) for fi in sys_cons.f]

print("States after DNA conservation laws:")
for x in sys_cons.x:
    print("  ", x)

print("\nDynamics after DNA conservation laws:")
for x, fi in zip(sys_cons.x, sys_cons.f):
    print(f"d{x}/dt =")
    pprint(simplify(fi))
    print()

Found conservation laws: [C_A1_D1 + C_R2_D1 + D1 - D1_tot, C_A2_D2 + C_R1_D2 + D2 - D2_tot]
States after DNA conservation laws:
   m1
   A1
   R1
   C_A1_D1
   m2
   A2
   R2
   C_A2_D2
   C_R2_D1
   C_R1_D2

Dynamics after DNA conservation laws:
dm1/dt =
C_A1_D1⋅ktx₁ - kdilₘ₁⋅m₁ - ktx₀ ₁⋅(C_A1_D1 + C_R2_D1 - D₁ ₜₒₜ)

dA1/dt =
A₁⋅kb₁⋅(C_A1_D1 + C_R2_D1 - D₁ ₜₒₜ) - A₁⋅kdil_A1 + C_A1_D1⋅ku₁ + ktl₁⋅m₁

dR1/dt =
C_R1_D2⋅ku_R1 + R₁⋅kb_R1⋅(C_A2_D2 + C_R1_D2 - D₂ ₜₒₜ) - R₁⋅kdil_R1 + ktl₁⋅m₁

dC_A1_D1/dt =
-A₁⋅kb₁⋅(C_A1_D1 + C_R2_D1 - D₁ ₜₒₜ) - C_A1_D1⋅ku₁

dm2/dt =
C_A2_D2⋅ktx₂ - kdilₘ₂⋅m₂ - ktx₀ ₂⋅(C_A2_D2 + C_R1_D2 - D₂ ₜₒₜ)

dA2/dt =
A₂⋅kb₂⋅(C_A2_D2 + C_R1_D2 - D₂ ₜₒₜ) - A₂⋅kdil_A2 + C_A2_D2⋅ku₂ + ktl₁⋅m₂

dR2/dt =
C_R2_D1⋅ku_R2 + R₂⋅kb_R2⋅(C_A1_D1 + C_R2_D1 - D₁ ₜₒₜ) - R₂⋅kdil_R2 + ktl₁⋅m₂

dC_A2_D2/dt =
-A₂⋅kb₂⋅(C_A2_D2 + C_R1_D2 - D₂ ₜₒₜ) - C_A2_D2⋅ku₂

dC_R2_D1/dt =
-C_R2_D1⋅ku_R2 - R₂⋅kb_R2⋅(C_A1_D1 + C_R2_D1 - D₁ ₜₒₜ)

dC_R1_D2/dt =
-C_R1_D2⋅ku_R1 - R₁⋅kb_R1⋅(C_A2_D2 + C_R1_D2 - D₂ ₜ

## Apply quasi-steady state assumptions

Some species, such as mRNA, degrade much faster than others, such as proteins. We can apply quasi-steady state assumptions to eliminate the fast species from the system. The `solve_timescale_separation` function can be used to eliminate fast species from the system. This AutoReduce method can find *all possible combinations* of fast species automatically by performing a combinatorial search over the system state-space and symbolically finding out whether a feasible solution is obtained. The user can also specify the fast species manually, as we do here for the toggle switch model. The fast species, as a first step, are the complex formations between the activator, repressor, and the DNA.

In [9]:
from autoreduce import solve_timescale_separation

slow_states_6 = [m1, A1, R1, m2, A2, R2]
fast_states_complexes = [C_A1_D1, C_R2_D1, C_A2_D2, C_R1_D2]

sys_6, fast_subsystem = solve_timescale_separation(
    sys_cons,
    slow_states=slow_states_6,
    fast_states=fast_states_complexes,
    debug=False,
)

sys_6.f = [simplify(fi) for fi in sys_6.f]

print("Six-state reduced system after promoter-complex QSS:")
for x, fi in zip(sys_6.x, sys_6.f):
    print(f"d{x}/dt =")
    pprint(simplify(fi))
    print()

print("Fast states collapsed:")
for x in fast_states_complexes:
    print("  ", x)

Successful solution obtained with states: [m1, A1, R1, m2, A2, R2]!
Six-state reduced system after promoter-complex QSS:
dm1/dt =
A₁⋅D₁ ₜₒₜ⋅kb₁⋅ktx₁⋅ku_R2 - A₁⋅kb₁⋅kdilₘ₁⋅ku_R2⋅m₁ + D₁ ₜₒₜ⋅ktx₀ ₁⋅ku₁⋅ku_R2 -  ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                         A₁⋅kb₁⋅ku_R2 + R₂⋅kb_R2⋅ku₁ + ku₁⋅ku_ ↪

↪ R₂⋅kb_R2⋅kdilₘ₁⋅ku₁⋅m₁ - kdilₘ₁⋅ku₁⋅ku_R2⋅m₁
↪ ────────────────────────────────────────────
↪ R2                                          

dA1/dt =
-A₁⋅kdil_A1 + ktl₁⋅m₁

dR1/dt =
-R₁⋅kdil_R1 + ktl₁⋅m₁

dm2/dt =
A₂⋅D₂ ₜₒₜ⋅kb₂⋅ktx₂⋅ku_R1 - A₂⋅kb₂⋅kdilₘ₂⋅ku_R1⋅m₂ + D₂ ₜₒₜ⋅ktx₀ ₂⋅ku₂⋅ku_R1 -  ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                         A₂⋅kb₂⋅ku_R1 + R₁⋅kb_R1⋅ku₂ + ku₂⋅ku_ ↪

↪ R₁⋅kb_R1⋅kdilₘ₂⋅ku₂⋅m₂ - kdilₘ₂⋅ku₂⋅ku_R1⋅m₂
↪ ────────────────────────────────────────────
↪ R1                                          

dA2/dt =
-A₂⋅kdil_A2 + kt

In [10]:
from sympy import cancel, factor, simplify, together, pprint, Eq, solve
from autoreduce import System
import numpy as np



def clean(expr):
    return factor(cancel(together(simplify(expr))))


sys_6.f = [clean(fi) for fi in sys_6.f]

print("Six-state model after promoter-complex QSS:")
for x, fi in zip(sys_6.x, sys_6.f):
    print(f"d{x}/dt =")
    pprint(clean(fi))
    print()

print("Parameters currently in sys_6:")
for p in sys_6.params:
    print("  ", p)

Six-state model after promoter-complex QSS:
dm1/dt =
-(-A₁⋅D₁ ₜₒₜ⋅kb₁⋅ktx₁⋅ku_R2 + A₁⋅kb₁⋅kdilₘ₁⋅ku_R2⋅m₁ - D₁ ₜₒₜ⋅ktx₀ ₁⋅ku₁⋅ku_R2 ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                            A₁⋅kb₁⋅ku_R2 + R₂⋅kb_R2⋅ku₁ + ku₁⋅ ↪

↪  + R₂⋅kb_R2⋅kdilₘ₁⋅ku₁⋅m₁ + kdilₘ₁⋅ku₁⋅ku_R2⋅m₁) 
↪ ─────────────────────────────────────────────────
↪ ku_R2                                            

dA1/dt =
-A₁⋅kdil_A1 + ktl₁⋅m₁

dR1/dt =
-R₁⋅kdil_R1 + ktl₁⋅m₁

dm2/dt =
-(-A₂⋅D₂ ₜₒₜ⋅kb₂⋅ktx₂⋅ku_R1 + A₂⋅kb₂⋅kdilₘ₂⋅ku_R1⋅m₂ - D₂ ₜₒₜ⋅ktx₀ ₂⋅ku₂⋅ku_R1 ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                            A₂⋅kb₂⋅ku_R1 + R₁⋅kb_R1⋅ku₂ + ku₂⋅ ↪

↪  + R₁⋅kb_R1⋅kdilₘ₂⋅ku₂⋅m₂ + kdilₘ₂⋅ku₂⋅ku_R1⋅m₂) 
↪ ─────────────────────────────────────────────────
↪ ku_R1                                            

dA2/dt =
-A₂⋅kdil_A2 + ktl₁⋅m₂

dR2/dt =
-R₂⋅kdil_R2 + ktl₁⋅m₂

Paramete

### Symbolic simplification of the reduced model: assume same expression rates

In [11]:
f6 = dict(zip(sys_6.x, sys_6.f))

expected_states = [m1, A1, R1, m2, A2, R2]
missing_states = [x for x in expected_states if x not in f6]


coexpression_rate_subs = {
    kdil_R1: kdil_A1,
    kdil_R2: kdil_A2,
}
coexpression_state_subs = {
    R1: A1,
    R2: A2,
}
f6_equal_rates = {
    x: clean(fi.subs(coexpression_rate_subs))
    for x, fi in f6.items()
}

D1_tot, D2_tot = symbols("D1_tot D2_tot")



x4 = [m1, A1, m2, A2]

f4 = [
    clean(f6_equal_rates[m1].subs(coexpression_state_subs)),
    clean(f6_equal_rates[A1].subs(coexpression_state_subs)),
    clean(f6_equal_rates[m2].subs(coexpression_state_subs)),
    clean(f6_equal_rates[A2].subs(coexpression_state_subs)),
]

params_4 = []
for p in sys_6.params:
    p_new = coexpression_rate_subs.get(p, p)
    if p_new not in params_4:
        params_4.append(p_new)

sys_4 = System(
    x=x4,
    f=f4,
    params=params_4,
    params_values=[sys_6.params_values[sys_6.params.index(p)] for p in params_4],
    x_init=[0.0, 0.0, 0.0, 0.0],
    C=np.array([[0, 1, 0, 0], [0, 0, 0, 1]]),
)

for fi in sys_4.f:
    if R1 in fi.free_symbols or R2 in fi.free_symbols:
        raise ValueError("R1 or R2 still appears after co-expression reduction.")

print("Four-equation model:")
for x, fi in zip(sys_4.x, sys_4.f):
    print(f"d{x}/dt =")
    pprint(clean(fi))
    print()

Four-equation model:
dm1/dt =
-(-A₁⋅D₁ ₜₒₜ⋅kb₁⋅ktx₁⋅ku_R2 + A₁⋅kb₁⋅kdilₘ₁⋅ku_R2⋅m₁ + A₂⋅kb_R2⋅kdilₘ₁⋅ku₁⋅m₁  ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                            A₁⋅kb₁⋅ku_R2 + A₂⋅kb_R2⋅ku₁ + ku₁⋅ ↪

↪ - D₁ ₜₒₜ⋅ktx₀ ₁⋅ku₁⋅ku_R2 + kdilₘ₁⋅ku₁⋅ku_R2⋅m₁) 
↪ ─────────────────────────────────────────────────
↪ ku_R2                                            

dA1/dt =
-A₁⋅kdil_A1 + ktl₁⋅m₁

dm2/dt =
-(A₁⋅kb_R1⋅kdilₘ₂⋅ku₂⋅m₂ - A₂⋅D₂ ₜₒₜ⋅kb₂⋅ktx₂⋅ku_R1 + A₂⋅kb₂⋅kdilₘ₂⋅ku_R1⋅m₂ - ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                           A₁⋅kb_R1⋅ku₂ + A₂⋅kb₂⋅ku_R1 + ku₂⋅k ↪

↪  D₂ ₜₒₜ⋅ktx₀ ₂⋅ku₂⋅ku_R1 + kdilₘ₂⋅ku₂⋅ku_R1⋅m₂) 
↪ ────────────────────────────────────────────────
↪ u_R1                                            

dA2/dt =
-A₂⋅kdil_A2 + ktl₁⋅m₂



### QSS: Fast transcription relative to translation 

In [12]:

slow_states_2 = [A1, A2]
fast_states_mrna = [m1, m2]

sys_2, mrna_fast_subsystem = solve_timescale_separation(
    sys_4,
    slow_states=slow_states_2,
    fast_states=fast_states_mrna,
    debug=False,
)

sys_2.f = [clean(fi) for fi in sys_2.f]

print("Two-state model after mRNA QSS:")
for x, fi in zip(sys_2.x, sys_2.f):
    print(f"d{x}/dt =")
    pprint(clean(fi))
    print()

Successful solution obtained with states: [A1, A2]!
Two-state model after mRNA QSS:
dA1/dt =
 ⎛  2                                                                          ↪
-⎝A₁ ⋅kb₁⋅kdil_A1⋅kdilₘ₁⋅ku_R2 + A₁⋅A₂⋅kb_R2⋅kdil_A1⋅kdilₘ₁⋅ku₁ - A₁⋅D₁ ₜₒₜ⋅kb ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                       kdilₘ₁⋅(A₁⋅kb₁⋅ku_R2 +  ↪

↪                                                                              ↪
↪ ₁⋅ktl₁⋅ktx₁⋅ku_R2 + A₁⋅kdil_A1⋅kdilₘ₁⋅ku₁⋅ku_R2 - D₁ ₜₒₜ⋅ktl₁⋅ktx₀ ₁⋅ku₁⋅ku_ ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪ A₂⋅kb_R2⋅ku₁ + ku₁⋅ku_R2)                                                    ↪

↪   ⎞ 
↪ R2⎠ 
↪ ────
↪     

dA2/dt =
 ⎛                                   2                                         ↪
-⎝A₁⋅A₂⋅kb_R1⋅kdil_A2⋅kdilₘ₂⋅ku₂ + A₂ ⋅kb₂⋅kdil_A2⋅kdilₘ₂⋅ku_R1 - A₂⋅D₂ ₜₒₜ⋅kb ↪
─────────────────────────────────────────────────────────

### Two-state reduced model

In [13]:
for x, fi in zip(sys_2.x, sys_2.f):
    print(f"d{x}/dt =")
    pprint(clean(fi))
    print()

dA1/dt =
 ⎛  2                                                                          ↪
-⎝A₁ ⋅kb₁⋅kdil_A1⋅kdilₘ₁⋅ku_R2 + A₁⋅A₂⋅kb_R2⋅kdil_A1⋅kdilₘ₁⋅ku₁ - A₁⋅D₁ ₜₒₜ⋅kb ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                       kdilₘ₁⋅(A₁⋅kb₁⋅ku_R2 +  ↪

↪                                                                              ↪
↪ ₁⋅ktl₁⋅ktx₁⋅ku_R2 + A₁⋅kdil_A1⋅kdilₘ₁⋅ku₁⋅ku_R2 - D₁ ₜₒₜ⋅ktl₁⋅ktx₀ ₁⋅ku₁⋅ku_ ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪ A₂⋅kb_R2⋅ku₁ + ku₁⋅ku_R2)                                                    ↪

↪   ⎞ 
↪ R2⎠ 
↪ ────
↪     

dA2/dt =
 ⎛                                   2                                         ↪
-⎝A₁⋅A₂⋅kb_R1⋅kdil_A2⋅kdilₘ₂⋅ku₂ + A₂ ⋅kb₂⋅kdil_A2⋅kdilₘ₂⋅ku_R1 - A₂⋅D₂ ₜₒₜ⋅kb ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                       kdilₘ

## So what? How can we use the reduced model?

The model derived above enables us to apply assume-guarantee contract based analysis to obtain guarantees on parameter regions that ensure bistability. This work (derivation of formal guarantees using the reduced models) was published in the IEEE Conference on Decision and Control (CDC) 2024 paper titled "Guaranteeing System-level Properties in Genetic Circuits Subject to Context Effects" by Inigo Incer; Ayush Pandey; Nicholas Nolan; Emma L. Peterman; Kate E. Galloway; Eduardo D. Sontag, Domitilla Del Vecchio.